### M2 SID 2026-2027
<div align="center">
  <h2 style="font-size: 36px;"><b>TP n°1 : web scraping</b></h2>
</div>

 
<div align="center">
  <h2 style="font-size: 30px;"><b>Par Muriel Visani</b></h2>
</div>

## Partie I: en autonomie

#### Lire le site suivant, pour mieux comprendre la manière dont BeautifulSoup fonctionne : https://cloro.dev/blog/beautifulsoup-web-scraping/

#### Faire le tutoriel suivant, qui vous permettra de découvrir la libraire BeautifulSoup de Python 
https://tech-insider.org/fr/tutoriel-web-scraping-python-extraction-donnees-2026/

In [1]:
!pip install -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'

[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Partie II: Web scraping - événements musicaux au CEPAC Silo, à Marseille

#### L'objet de cette partie est de vous faire :

> #### - Récupérer les données relatives aux événements culturels au CEPAC Silo à Marseille (concerts + différents spectacles : humour, danse, etc.). Le lien de cette page est "https://www.cepacsilo-marseille.fr/evenements/"

> #### - Structurer ces données

> #### - Placer ces données dans un dataframe Pandas et mener une analyse exploratoire sommaire des données, c.à.d. calculer quelques statistiques de base sur ces événements (prix moyen, etc.)

Parfois, dans ce TP, je vous donne la réponse attendue, selon le contenu de la page lorsque j'ai monté ce TP (lors de l'été 2026).

Le cas échéant, si la page du CEPAC a trop changé depuis que j'ai monté ce TP, alors je vous donne son contenu dans le répertoire cepac_silo_pages.zip.

Vous pouvez travailler sur ce répertoire zippé pour vérifier que vous avez bien la bonne réponse.

#### Q0
Importez les librairies nécessaires

In [2]:
import requests
import os
import httpx
import csv
import pandas as pd
from bs4 import BeautifulSoup

#### Q1 
Avec les outils de développement de Chrome (et le cas échéant un éditeur de code), regardez bien attentivement le code HTM de la page "https://www.cepacsilo-marseille.fr/evenements/", testez les liens URL qui s'y trouvent, etc. C'est une étape cruciale.

Il s'agit que vous compreniez comment est structurée la page, où se trouvent les données importantes, etc.

Dans ce TP, on va s'intéresser aux informations concernant les événements musicaux d'Aix-Marseille, à savoir :

> - le type d'événement (concert, spectacle, one man show, humour, etc.)

> - la date de l'événement : nom du jour (lundi, mardi, etc), numéro du jour (01 comme dans 1er Octobre par exemple), le nom du mois, l'année

> - l'heure de l'événement 

> - l'artiste, ou la liste des artistes, qui se produira sur scène

> - le prix d'une place, soit en catégorie 2, soit en catégorie unique, soit en catégorie debout (pour cela, il faut normalement cliquer sur "Réserver")

Tentez de repérer, dans cette page HTML, où se trouvent ces informations, et donnez quelques éléments de réponse (balises où ces événement se trouvent) dans la cellule Markdown ci-après.

Dans cette page les informations sont entre différentes balises à savoir pour :

Type événement : une balise span
date d'événement : une balise p
l'artiste :h3
pour l'heure une fois que l'on clique sur l'évenement on le retrouve dans une balise span
et enfin pour le prix on doit chercher dans la classe category price ou on va trouver les prix entre des balises div

#### Q2
Avec httpx, récupérez automatiquement le nombre de pages de "https://www.cepacsilo-marseille.fr/evenements/" (normalement 10 en tout). 

Repérez les URLs de ces pages, et ce qui s'affiche quand vous allez une page trop loin.

In [3]:


url = "https://www.cepacsilo-marseille.fr/evenements/"

response = httpx.get(url)
response.raise_for_status() # to ensure we got a successful response

soup = BeautifulSoup(response.text, "lxml")

pages = soup.find_all(
    "a",
    class_="events-archive__pagination-item"
)

numeros_pages = []

for page in pages:
    numero = page.get("data-page")

    numeros_pages.append(int(numero))

nombre_pages = max(numeros_pages)
print("Nombre de pages :", nombre_pages)


Nombre de pages : 10


#### Q3
Avec httpx, récupérez tout le contenu textuel de toutes les pages de "https://www.cepacsilo-marseille.fr/evenements/", et mettez-le dans une liste appelée "pages".
Vérifiez que les status_code commencent bien par 2 (ce qui signifie que les pages ont bien été scrapées).

In [4]:
pages = []

for numero in range(1, nombre_pages + 1):

    url = f"https://www.cepacsilo-marseille.fr/evenements/page/{numero}/"

    response = httpx.get(url, follow_redirects=True)

    print(f"Page {numero} : {response.status_code}")

    if str(response.status_code).startswith("2"):
        pages.append(response.text)

Page 1 : 200
Page 2 : 200
Page 3 : 200
Page 4 : 200
Page 5 : 200
Page 6 : 200
Page 7 : 200
Page 8 : 200
Page 9 : 200
Page 10 : 200


#### Q4

Toujours avec httpx, sauvergardez le contenu HTML de l'ensemble des pages d'événements (un fichier HTML par page), dans un répertoire nommé "TP_cepac_silo_pages".

In [5]:



os.makedirs("TP_cepac_silo_pages", exist_ok=True)


for i in range(len(pages)):

    num_page = f"TP_cepac_silo_pages/page_{i + 1}.html"

    with open(num_page, "w", encoding="utf-8") as fichier:
        fichier.write(pages[i])


#### Q5
Avec BeautifulSoup, placez le contenu HTML de "pages" (toutes les pages HTML) dans une liste appelée "soup".

In [6]:


soup = []

for page in pages:
    contenu = BeautifulSoup(page, "lxml")
    soup.append(contenu)

#print(soup)

#### Q6
Retournez (avec print) le titre du premier événement mentionné dans la première page. Le type de sortie attendue est par exemple :


Alizée

In [7]:
titre = soup[0].find_all("h3", class_="card-event__title")

print(titre[0].text)

DEVIN TOWNSEND


#### Q7
## <font color='red'> Point de validation 1 : une fois que vous aurez terminé cette question, appelez la prof et attendez-vous à répondre à des questions sur votre code <img src="buttonValidate.gif"> 

Pour tous les événements mentionnés dans la première page, retournez (avec print) le titre de cet événement, précédé du mot-clé "Événement" et de l'ordre d'apparition de cet événement dans la page (exemple "Événement 1"). Le type de sortie attendue est par exemple :

Événement 1: Alizée

Événement 2: MAC DEMARCO

Événement 3: DEVIN TOWNSEND

Événement 4: The bootleg beatles

Événement 5: LAURENT FEBVAY

Événement 6: LETZ ZEP

Événement 7: THE SISTERS OF MERCY

Événement 8: ORCHESTRE CURIEUX

Événement 9: BREL !

Événement 10: Marie Benoliel

Événement 11: TIF

In [8]:
titres =[]
for s in soup:
  titre = s.find_all("h3", class_="card-event__title")
  titres.append(titre)

# print all titles of first page
page_1_titles = [t.text for t in titres[0]]
print(page_1_titles)

['DEVIN TOWNSEND', 'The bootleg beatles', 'LAURENT FEBVAY', 'LETZ ZEP', 'THE SISTERS OF MERCY', 'BREL !', 'Marie Benoliel', 'TIF', 'Boudchart', 'WORLDS APART', 'Amine Radi']


#### Q8
Pour tous les événements mentionnés dans la première page, précisez le type d'événement (Concert, one man show, humour, etc). Le type de sortie attendue est par exemple :

```Événement 1 : Alizée  -  Concert
Événement 2 : MAC DEMARCO  -  Concert
Événement 3 : DEVIN TOWNSEND  -  Concert
Événement 4 : The bootleg beatles  -  Concert
Événement 5 : LAURENT FEBVAY  -  One man show
Événement 6 : LETZ ZEP  -  Concert
Événement 7 : THE SISTERS OF MERCY  -  Concert
Événement 8 : ORCHESTRE CURIEUX  -  Spectacle
Événement 9 : BREL !  -  Spectacle
Événement 10 : Marie Benoliel  -  Spectacle
Événement 11 : TIF  -  Concert - Rap / Urbain


In [9]:
# type of event is inside span class="tag-pill"
types = []
for s in soup:
  type_event = s.find_all("span", class_="tag-pill")
  types.append(type_event)

# print all types of events of first page with respective titles
page_1_types = [t.text for t in types[0]]
page_1_titles = [t.text for t in titres[0]]
for i in range(len(page_1_titles)):
    print(f"Événement {i+1}: {page_1_titles[i]} - Type: {page_1_types[i]}")


Événement 1: DEVIN TOWNSEND - Type: Concert
Événement 2: The bootleg beatles - Type: Concert
Événement 3: LAURENT FEBVAY - Type: One man show
Événement 4: LETZ ZEP - Type: Concert
Événement 5: THE SISTERS OF MERCY - Type: Concert
Événement 6: BREL ! - Type: Spectacle
Événement 7: Marie Benoliel - Type: Spectacle
Événement 8: TIF - Type: Concert - Rap / Urbain
Événement 9: Boudchart - Type: Concert
Événement 10: WORLDS APART - Type: Concert
Événement 11: Amine Radi - Type: Humour


#### Q9
Pour tous les événements mentionnés dans la première page, précisez la ou les date(s) de l'événement, comme par exemple :

```Événement 1 : Alizée  -  Concert
   Le  Jeudi 10 septembre 2026 - 20:00
Événement 2 : MAC DEMARCO  -  Concert
   Le  Dimanche 13 septembre 2026 - 20:00
   Le  Lundi 14 septembre 2026 - 20:00
Événement 3 : DEVIN TOWNSEND  -  Concert
   Le  Mercredi 23 septembre 2026 - 20:00
Événement 4 : The bootleg beatles  -  Concert
   Le  Jeudi 24 septembre 2026 - 20:00
Événement 5 : LAURENT FEBVAY  -  One man show
   Le  Samedi 26 septembre 2026 - 20:30
Événement 6 : LETZ ZEP  -  Concert
   Le  Mercredi 30 septembre 2026 - 20:00
Événement 7 : THE SISTERS OF MERCY  -  Concert
   Le  Jeudi 01 octobre 2026 - 20:00
Événement 8 : ORCHESTRE CURIEUX  -  Spectacle
   Le  Vendredi 02 octobre 2026 - 20:00
Événement 9 : BREL !  -  Spectacle
   Le  Dimanche 04 octobre 2026 - 17:00
Événement 10 : Marie Benoliel  -  Spectacle
   Le  Mercredi 07 octobre 2026 - 20:00
Événement 11 : TIF  -  Concert - Rap / Urbain
   Le  Jeudi 08 octobre 2026 - 20:00```

In [10]:
evenements = soup[0].find_all(class_="card-event")

compteur = 1

for evenement in evenements:

    titre = evenement.find("h3", class_="card-event__title")
    type_event = evenement.find("span", class_="tag-pill")
    date = evenement.find("p", class_="card-event__date")

    print(f"Événement {compteur}: {titre.text} - {type_event.text}")
    print(f"   Le {date.text}")

    compteur = compteur + 1

Événement 1: DEVIN TOWNSEND - Concert
   Le Mercredi 23 septembre 2026
Événement 2: The bootleg beatles - Concert
   Le Jeudi 24 septembre 2026
Événement 3: LAURENT FEBVAY - One man show
   Le Samedi 26 septembre 2026
Événement 4: LETZ ZEP - Concert
   Le Mercredi 30 septembre 2026
Événement 5: THE SISTERS OF MERCY - Concert
   Le Jeudi 1 octobre 2026
Événement 6: BREL ! - Spectacle
   Le Dimanche 4 octobre 2026
Événement 7: Marie Benoliel - Spectacle
   Le Mercredi 7 octobre 2026
Événement 8: TIF - Concert - Rap / Urbain
   Le Jeudi 8 octobre 2026
Événement 9: Boudchart - Concert
   Le Vendredi 9 octobre 2026
Événement 10: WORLDS APART - Concert
   Le Samedi 10 octobre 2026
Événement 11: Amine Radi - Humour
   Le Samedi 17 octobre 2026


#### Q10
Même question, mais pour tous les CONCERTS mentionnés dans la page (mais pas pour les autres événements).

Le type de sortie attendue est par exemple :

```Événement 1 : Alizée  -  Concert
   Le  Jeudi 10 septembre 2026 - 20:00
Événement 2 : MAC DEMARCO  -  Concert
   Le  Dimanche 13 septembre 2026 - 20:00
   Le  Lundi 14 septembre 2026 - 20:00
Événement 3 : DEVIN TOWNSEND  -  Concert
   Le  Mercredi 23 septembre 2026 - 20:00
Événement 4 : The bootleg beatles  -  Concert
   Le  Jeudi 24 septembre 2026 - 20:00
Événement 5 : LETZ ZEP  -  Concert
   Le  Mercredi 30 septembre 2026 - 20:00
Événement 6 : THE SISTERS OF MERCY  -  Concert
   Le  Jeudi 01 octobre 2026 - 20:00
Événement 7 : TIF  -  Concert - Rap / Urbain
   Le  Jeudi 08 octobre 2026 - 20:00```

In [11]:


compteur = 1

for evenement in evenements:

    type_event = evenement.find(class_="tag-pill")

    if "Concert" in type_event.text:

        titre = evenement.find("h3", class_="card-event__title")
        date = evenement.find("p", class_="card-event__date")

        print(f"Événement {compteur}: {titre.text} - Concert")
        print(f"   Le {date.text}")

        compteur = compteur + 1

Événement 1: DEVIN TOWNSEND - Concert
   Le Mercredi 23 septembre 2026
Événement 2: The bootleg beatles - Concert
   Le Jeudi 24 septembre 2026
Événement 3: LETZ ZEP - Concert
   Le Mercredi 30 septembre 2026
Événement 4: THE SISTERS OF MERCY - Concert
   Le Jeudi 1 octobre 2026
Événement 5: TIF - Concert
   Le Jeudi 8 octobre 2026
Événement 6: Boudchart - Concert
   Le Vendredi 9 octobre 2026
Événement 7: WORLDS APART - Concert
   Le Samedi 10 octobre 2026


#### Q11

## <font color='red'> Point de validation 2 : une fois que vous aurez terminé cette question, appelez la prof et attendez-vous à répondre à des questions sur votre code <img src="buttonValidate.gif"> 

Même question, mais pour tous les EVENEMENTS mentionnés dans TOUTES LES pages.

Conseil : mettre les événements dans une liste Python, et utiliser une boucle pour "parser" chacun de ces événements un à un, en n'oubliant pas de mettre à None les éventuels éléments vides. Cela vous facilitera considérablement le travail pour les questions suivantes !


La sortie attendue devra mentionner tous les éléments ci-dessous et respecter la même mise en forme :

```Événement 1 : Alizée  -  Concert
   Le  Jeudi 10 septembre 2026 - 20:00
Événement 2 : MAC DEMARCO  -  Concert
   Le  Dimanche 13 septembre 2026 - 20:00
   Le  Lundi 14 septembre 2026 - 20:00
Événement 3 : DEVIN TOWNSEND  -  Concert
   Le  Mercredi 23 septembre 2026 - 20:00
Événement 4 : The bootleg beatles  -  Concert
   Le  Jeudi 24 septembre 2026 - 20:00
Événement 5 : LAURENT FEBVAY  -  One man show
   Le  Samedi 26 septembre 2026 - 20:30
Événement 6 : LETZ ZEP  -  Concert
   Le  Mercredi 30 septembre 2026 - 20:00
Événement 7 : THE SISTERS OF MERCY  -  Concert
   Le  Jeudi 01 octobre 2026 - 20:00
Événement 8 : ORCHESTRE CURIEUX  -  Spectacle
   Le  Vendredi 02 octobre 2026 - 20:00
Événement 9 : BREL !  -  Spectacle
   Le  Dimanche 04 octobre 2026 - 17:00
Événement 10 : Marie Benoliel  -  Spectacle
   Le  Mercredi 07 octobre 2026 - 20:00
Événement 11 : TIF  -  Concert - Rap / Urbain
   Le  Jeudi 08 octobre 2026 - 20:00
Événement 12 : WORLDS APART  -  Concert
   Le  Samedi 10 octobre 2026 - 20:00
Événement 13 : Amine Radi  -  Humour
   Le  Samedi 17 octobre 2026 - 20:30
Événement 14 : TINA ARENA  -  Concert
   Le  Mercredi 21 octobre 2026 - 20:00
Événement 15 : THE VOICE OF GEORGE MICHAEL  -  Concert
   Le  Jeudi 22 octobre 2026 - 20:30
   
...

Événement 98 : NASH  -  Humour
   Le  Vendredi 05 novembre 2027 - 20:30
Événement 99 : ARNAUD TSAMERE  -  Humour
   Le  Samedi 20 novembre 2027 - 20:00
Événement 100 : JULIEN ROSSIER EST COLUCHE !  -  Spectacle
   Le  Jeudi 25 novembre 2027 - 20:00
Événement 101 : SOFIA BELABBES  -  Humour
   Le  Vendredi 10 décembre 2027 - 20:00
Événement 102 : LES DIX COMMANDEMENTS  -  Spectacle
   Le  Jeudi 24 février 2028 - 20:30
Événement 103 : PIAF ! LE SPECTACLE  -  Spectacle
   Le  Mardi 29 février 2028 - 20:30





In [12]:
evenements = []

for page in soup:
    evenements_page = page.find_all(class_="card-event")

    for evenement in evenements_page:
        evenements.append(evenement)
#jusqu'ici j'ai recupéré les evenements de toutes les pages que javais dans soup que j'ai mtn dans evenement


compteur = 1

for evenement in evenements:

    titre = evenement.find("h3", class_="card-event__title")
    types = evenement.find_all("span", class_="tag-pill")

    lien = evenement.find("a", class_="card-event__overlay-link") # here we are looking for the link to the event's detail page

    if lien is None:
        url_evenement = None
    else:
        url_evenement = lien.get("href")


    print(f"Événement {compteur} : {titre.text}", end="")

    for type_event in types:
        print(f" - {type_event.text}", end="")

    print()

    if url_evenement is not None:

        response = httpx.get(url_evenement)

        soup_evenement = BeautifulSoup(response.text, "lxml")

        sessions = soup_evenement.find_all("li", class_="bl-he__session")


        for session in sessions:

            date = session.find("span", class_="bl-he__session-date")
            heure = session.find("span", class_="bl-he__session-time")

            if date is None:
                date_texte = None
            else:
                date_texte = date.text

            if heure is None:
                heure_texte = None
            else:
                heure_texte = heure.text

            if date_texte is not None and heure_texte is not None:
                print(f"   Le {date_texte} - {heure_texte}")

    compteur = compteur + 1

Événement 1 : DEVIN TOWNSEND - Concert
   Le mercredi 23 septembre 2026 - 20h00
Événement 2 : The bootleg beatles - Concert
Événement 3 : LAURENT FEBVAY - One man show
   Le samedi 26 septembre 2026 - 20h30
Événement 4 : LETZ ZEP - Concert
   Le mercredi 30 septembre 2026 - 20h00
Événement 5 : THE SISTERS OF MERCY - Concert
   Le jeudi 01 octobre 2026 - 20h00
Événement 6 : BREL ! - Spectacle
   Le dimanche 04 octobre 2026 - 17h00
Événement 7 : Marie Benoliel - Spectacle
   Le mercredi 07 octobre 2026 - 20h00
Événement 8 : TIF - Concert - Rap / Urbain
   Le jeudi 08 octobre 2026 - 20h00
Événement 9 : Boudchart - Concert
   Le vendredi 09 octobre 2026 - 20h00
Événement 10 : WORLDS APART - Concert
   Le samedi 10 octobre 2026 - 20h00
Événement 11 : Amine Radi - Humour
   Le samedi 17 octobre 2026 - 20h30
Événement 12 : TINA ARENA - Concert
   Le mercredi 21 octobre 2026 - 20h00
Événement 13 : THE VOICE OF GEORGE MICHAEL - Concert
   Le jeudi 12 février 2026 - 20h30
   Le jeudi 22 octobre 

### Q12

## <font color='red'> Point de validation 3 : une fois que vous aurez terminé cette question, appelez la prof et attendez-vous à répondre à des questions sur votre code <img src="buttonValidate.gif"> 


Pour chaque événement, rajouter le lien de réservation : celui où on peut trouver le prix des types de places que l'on veut (debout, catégorie 2 ou catégorie unique). Lorsque ce lien n'est pas donné (par exemple parce que le concert est complet), imprimer "Réservation impossible".

Mettre toutes ces informations (titre, catégorie, dates / heures, liens de réservation ou "Réservation impossible") dans un même dictionnaire de données. Cela vous facilitera beaucoup la tâche pour la suite...
    
N.B. Vous pourriez aussi mettre cela dans une liste, ça serait peut-être même plus facile à gérer, mais je veux vous faire travailler les dictionnaires et leur lien avec pandas...

In [13]:
donnees = {
    "titre": [],
    "categorie": [],
    "dates_heures": [],
    "reservation": []
}


for evenement in evenements:



    titre = evenement.find("h3", class_="card-event__title")
    titre_texte = titre.text


    categorie = evenement.find("span", class_="tag-pill")
    categorie_texte = categorie.text




    lien = evenement.find("a", class_="card-event__overlay-link")
    url_evenement = lien.get("href")



    dates_heures = []
    reservation = "Réservation impossible"




    if url_evenement is not None:

        response = httpx.get(url_evenement)

        soup_evenement = BeautifulSoup(response.text, "lxml")



        sessions = soup_evenement.find_all(
            "li",
            class_="bl-he__session"
        )

        for session in sessions:

            date = session.find(
                "span",
                class_="bl-he__session-date"
            )

            heure = session.find(
                "span",
                class_="bl-he__session-time"
            )

            if date is None:
                date_texte = None
            else:
                date_texte = date.text

            if heure is None:
                heure_texte = None
            else:
                heure_texte = heure.text


            if date_texte is not None and heure_texte is not None:

                date_heure = date_texte + " - " + heure_texte

                dates_heures.append(date_heure)




        lien_reservation = soup_evenement.find(
            "a",
            class_="modal-booking__cta"
        )

        if lien_reservation is not None:
            reservation = lien_reservation.get("href")


    donnees["titre"].append(titre_texte)

    donnees["categorie"].append(categorie_texte)

    donnees["dates_heures"].append(dates_heures)

    donnees["reservation"].append(reservation)




In [14]:
compteur = 1

for i in range(5):

    print(f"Événement {compteur} : {donnees['titre'][i]} - {donnees['categorie'][i]}")

    for date_heure in donnees["dates_heures"][i]:
        print(f"   Le {date_heure}")

    print(f"   Réservation : {donnees['reservation'][i]}")

    compteur = compteur + 1

Événement 1 : DEVIN TOWNSEND - Concert
   Le mercredi 23 septembre 2026 - 20h00
   Réservation : https://billetterie.cepacsilo-marseille.fr/fr/product/911/le_cepac_silo/devin_townsend
Événement 2 : The bootleg beatles - Concert
   Réservation : https://billetterie.cepacsilo-marseille.fr/fr/product/890/le_cepac_silo/the_bootleg_beatles
Événement 3 : LAURENT FEBVAY - One man show
   Le samedi 26 septembre 2026 - 20h30
   Réservation : https://billetterie.cepacsilo-marseille.fr/fr/product/665/le_cepac_silo/laurent_febvay
Événement 4 : LETZ ZEP - Concert
   Le mercredi 30 septembre 2026 - 20h00
   Réservation : https://billetterie.cepacsilo-marseille.fr/fr/product/911/le_cepac_silo/devin_townsend
Événement 5 : THE SISTERS OF MERCY - Concert
   Le jeudi 01 octobre 2026 - 20h00
   Réservation : https://billetterie.cepacsilo-marseille.fr/fr/product/845/le_cepac_silo/the_sisters_of_mercy


In [17]:
# print first event where: reservation = "Réservation impossible"
assumption = False
i=1
while assumption == False and i < len(donnees['reservation']):
    if donnees['reservation'][i] == "Réservation impossible":
        print(f"Événement: {donnees['titre'][i]} - {donnees['categorie'][i]}, Réservation: {donnees['reservation'][i]}")
        assumption = True
    i += 1

Événement: DRAG RACE FRANCE LIVE - Comédie musicale, Réservation: Réservation impossible


#### Q13

## <font color='red'> Point de validation 4 : une fois que vous aurez terminé cette question, appelez la prof et attendez-vous à répondre à des questions sur votre code <img src="buttonValidate.gif"> 


Récupérer, dans le même dictionnaire python, le prix de chaque place. Il existe généralement plusieurs catégories réservables mais, selon les événements / salles, toutes les catégories ne sont pas forcément réservables. 
    
Pour chaque événement, vous récupérerez donc les prix - par ordre de priorité - de la "catégorie 2", puis de la "catégorie assis- placement libre", puis de la "catégorie unique", ou bien enfin de la "fosse". 
Si plusieurs prix sont possibles pour la catégorie que vous avez sélectionnée, alors prenez le prix minimum.

Lorsque la catégorie existe mais que vous ne pouvez pas accéder au prix car le concert est complet, remplacez le prix par None (puisque vous ne pouvez pas le connaître a priori).

Attention : vous devez gérer automatiquement les redirections de pages, quand c'est le cas.

In [18]:
##Je me suis permis de changer un peu les categories car dans le site il y categorie 1,2 ou OR mais pas fosse ou categorie unique

import httpx
from bs4 import BeautifulSoup


donnees = {
    "titre": [],
    "categorie": [],
    "dates_heures": [],
    "reservation": [],
    "categorie_2": [],
    "prix_categorie_2": [],
    "categorie_1": [],
    "prix_categorie_1": [],
    "categorie_or": [],
    "prix_categorie_or": []
}


for evenement in evenements:

    titre = evenement.find(
        "h3",
        class_="card-event__title"
    )

    if titre is None:
        titre_texte = None
    else:
        titre_texte = titre.text.strip()


    categorie = evenement.find(
        "span",
        class_="tag-pill"
    )

    if categorie is None:
        categorie_texte = None
    else:
        categorie_texte = categorie.text.strip()


    lien = evenement.find(
        "a",
        class_="card-event__overlay-link"
    )

    if lien is None:
        url_evenement = None
    else:
        url_evenement = lien.get("href")


    dates_heures = []

    reservation = "Réservation impossible"

    categorie_2 = None
    prix_categorie_2 = None

    categorie_1 = None
    prix_categorie_1 = None

    categorie_or = None
    prix_categorie_or = None


    if url_evenement is not None:

        response = httpx.get(
            url_evenement,
            follow_redirects=True
        )

        soup_evenement = BeautifulSoup(
            response.text,
            "lxml"
        )


        sessions = soup_evenement.find_all(
            "li",
            class_="bl-he__session"
        )

        for session in sessions:

            date = session.find(
                "span",
                class_="bl-he__session-date"
            )

            heure = session.find(
                "span",
                class_="bl-he__session-time"
            )

            if date is None:
                date_texte = None
            else:
                date_texte = date.text.strip()

            if heure is None:
                heure_texte = None
            else:
                heure_texte = heure.text.strip()

            if date_texte is not None and heure_texte is not None:

                date_heure = (
                    date_texte
                    + " - "
                    + heure_texte
                )

                dates_heures.append(
                    date_heure
                )


        lien_reservation = soup_evenement.find(
            "a",
            class_="modal-booking__cta"
        )

        if lien_reservation is not None:

            reservation = lien_reservation.get("href")


            response_reservation = httpx.get(
                reservation,
                follow_redirects=True
            )

            soup_reservation = BeautifulSoup(
                response_reservation.text,
                "lxml"
            )


            categories_places = soup_reservation.find_all(
                "div",
                class_="category-wrapper"
            )


            for categorie_place in categories_places:

                titre_place = categorie_place.find(
                    "div",
                    class_="title"
                )

                if titre_place is not None:

                    texte_categorie = titre_place.text.strip()
                    texte_categorie_min = texte_categorie.lower()


                    prix_balises = categorie_place.find_all(
                        "div",
                        class_="category-price"
                    )

                    prix_trouves = []


                    for prix_balise in prix_balises:

                        prix_texte = prix_balise.text.strip()

                        prix_texte = prix_texte.replace(
                            "€",
                            ""
                        )

                        prix_texte = prix_texte.replace(
                            "\xa0",
                            ""
                        )

                        prix_texte = prix_texte.replace(
                            ",",
                            "."
                        )

                        prix_texte = prix_texte.strip()


                        try:

                            prix_nombre = float(
                                prix_texte
                            )

                            prix_trouves.append(
                                prix_nombre
                            )

                        except:

                            pass


                    if len(prix_trouves) > 0:

                        prix_minimum = min(
                            prix_trouves
                        )

                    else:

                        prix_minimum = None



                    if "categorie 2" in texte_categorie_min:

                        categorie_2 = texte_categorie
                        prix_categorie_2 = prix_minimum


                    if "categorie 1" in texte_categorie_min:

                        categorie_1 = texte_categorie
                        prix_categorie_1 = prix_minimum


                    if "categorie or" in texte_categorie_min:

                        categorie_or = texte_categorie
                        prix_categorie_or = prix_minimum


    donnees["titre"].append(
        titre_texte
    )

    donnees["categorie"].append(
        categorie_texte
    )

    donnees["dates_heures"].append(
        dates_heures
    )

    donnees["reservation"].append(
        reservation
    )

    donnees["categorie_2"].append(
        categorie_2
    )

    donnees["prix_categorie_2"].append(
        prix_categorie_2
    )

    donnees["categorie_1"].append(
        categorie_1
    )

    donnees["prix_categorie_1"].append(
        prix_categorie_1
    )

    donnees["categorie_or"].append(
        categorie_or
    )

    donnees["prix_categorie_or"].append(
        prix_categorie_or
    )

In [19]:
compteur = 1

for i in range(len(donnees["titre"])):

    print(
        f"Événement {compteur} : "
        f"{donnees['titre'][i]}"
    )

    print(
        "Catégorie de l'événement :",
        donnees["categorie"][i]
    )


    for date_heure in donnees["dates_heures"][i]:

        print(
            "   Le",
            date_heure
        )


    print(
        "Réservation :",
        donnees["reservation"][i]
    )


    print(
        "Catégorie 2 :",
        donnees["categorie_2"][i]
    )

    print(
        "Prix catégorie 2 :",
        donnees["prix_categorie_2"][i]
    )


    print(
        "Catégorie 1 :",
        donnees["categorie_1"][i]
    )

    print(
        "Prix catégorie 1 :",
        donnees["prix_categorie_1"][i]
    )


    print(
        "Catégorie OR :",
        donnees["categorie_or"][i]
    )

    print(
        "Prix catégorie OR :",
        donnees["prix_categorie_or"][i]
    )


    print()

    compteur = compteur + 1

Événement 1 : DEVIN TOWNSEND
Catégorie de l'événement : Concert
   Le mercredi 23 septembre 2026 - 20h00
Réservation : https://billetterie.cepacsilo-marseille.fr/fr/product/911/le_cepac_silo/devin_townsend
Catégorie 2 : CATEGORIE 2
Prix catégorie 2 : 63.0
Catégorie 1 : CATEGORIE 1
Prix catégorie 1 : 74.0
Catégorie OR : CATEGORIE OR
Prix catégorie OR : 90.5

Événement 2 : The bootleg beatles
Catégorie de l'événement : Concert
Réservation : https://billetterie.cepacsilo-marseille.fr/fr/product/890/le_cepac_silo/the_bootleg_beatles
Catégorie 2 : None
Prix catégorie 2 : None
Catégorie 1 : None
Prix catégorie 1 : None
Catégorie OR : None
Prix catégorie OR : None

Événement 3 : LAURENT FEBVAY
Catégorie de l'événement : One man show
   Le samedi 26 septembre 2026 - 20h30
Réservation : https://billetterie.cepacsilo-marseille.fr/fr/product/665/le_cepac_silo/laurent_febvay
Catégorie 2 : CATEGORIE 2
Prix catégorie 2 : 29.0
Catégorie 1 : CATEGORIE 1
Prix catégorie 1 : 39.0
Catégorie OR : None
Prix

#### Q14
On veut maintenant mettre toutes ces informations dans un dataframe de la librairie pandas nommé "evenements": https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html

Pour ceux qui ne connaîtraient toujours pas la librairie pandas, elle nous permet de présenter des données de manière facilement lisible, et de calculer des statistiques basiques sur ces données.

Une fois tous les champs bien vérifiés, vous pouvez supprimer les colonnes concernant le numéro de l'événement, les dates "complètes" (mais pas les noms de jours etc), et les liens de réservation.

La sortie attendue est du type :

N.B. Si on avait utilisé une liste au lieu d'un dictionnaire, on aurait a priori obtenu une seule colonne pour "Prix" par exemple (donc moins de NaN), mais vous auriez plusieurs lignes pour un même "titre" (souvent, l'artiste)... Ici, il y a au plus 3 dates pour chaque "titre", donc les deux solutions restent acceptables. 

<img src="Capture d’écran 2026-07-24 à 15.00.29.png"> 


In [20]:
import pandas as pd


evenements = pd.DataFrame(donnees)


evenements["jour_1"] = None
evenements["heure_1"] = None

evenements["jour_2"] = None
evenements["heure_2"] = None

evenements["jour_3"] = None
evenements["heure_3"] = None


for i in range(len(evenements)):

    dates = evenements.loc[i, "dates_heures"]


    if len(dates) >= 1:

        morceaux = dates[0].split(" - ")

        date = morceaux[0]
        heure = morceaux[1]

        jour = date.split()[0]

        evenements.loc[i, "jour_1"] = jour
        evenements.loc[i, "heure_1"] = heure


    if len(dates) >= 2:

        morceaux = dates[1].split(" - ")

        date = morceaux[0]
        heure = morceaux[1]

        jour = date.split()[0]

        evenements.loc[i, "jour_2"] = jour
        evenements.loc[i, "heure_2"] = heure


    if len(dates) >= 3:

        morceaux = dates[2].split(" - ")

        date = morceaux[0]
        heure = morceaux[1]

        jour = date.split()[0]

        evenements.loc[i, "jour_3"] = jour
        evenements.loc[i, "heure_3"] = heure


evenements = evenements.drop(
    columns=[
        "dates_heures",
        "reservation"
    ]
)


display(evenements)

,titre,categorie,categorie_2,prix_categorie_2,categorie_1,prix_categorie_1,categorie_or,prix_categorie_or,jour_1,heure_1,jour_2,heure_2,jour_3,heure_3
0,DEVIN TOWNSEND,Concert,CATEGORIE 2,63.0,CATEGORIE 1,74.0,CATEGORIE OR,90.5,mercredi,20h00,None,None,None,None
1,The bootleg beatles,Concert,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None,None
2,LAURENT FEBVAY,One man show,CATEGORIE 2,29.0,CATEGORIE 1,39.0,NaN,NaN,samedi,20h30,None,None,None,None
3,LETZ ZEP,Concert,CATEGORIE 2,63.0,CATEGORIE 1,74.0,CATEGORIE OR,90.5,mercredi,20h00,None,None,None,None
4,THE SISTERS OF MERCY,Concert,NaN,NaN,NaN,NaN,NaN,NaN,jeudi,20h00,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,BLANDINE LEHOUT,Humour,CATEGORIE 2,36.0,CATEGORIE 1,39.0,NaN,NaN,vendredi,20h00,None,None,None,None
106,SOFIA BELABBES,Humour,CATEGORIE 2,32.0,CATEGORIE 1,35.0,NaN,NaN,vendredi,20h00,None,None,None,None
107,LES DIX COMMANDEMENTS,Spectacle,CATEGORIE 2,49.0,CATEGORIE 1,64.0,CATEGORIE OR,74.0,jeudi,20h30,None,None,None,None
108,YANN MARGUET,Humour,CATEGORIE 2,36.0,CATEGORIE 1,39.0,NaN,NaN,samedi,20h00,None,None,None,None


#### Q15

## <font color='red'> Point de validation 5 : une fois que vous aurez terminé cette question et vérifié avec le répertoire cepac_silo_pages donné sur Ametice que vous obtenez la bonne réponse, appelez la prof et attendez-vous à répondre à des questions sur votre code <img src="buttonValidate.gif"> 

Mettre tous les concerts du dataframe "evenement" dans un dataframe "concert".
Mettre tous les spectacles humour ou one-man-show du dataframe "evenement" dans un dataframe "comedie".

Pour répondre aux questions suivantes, utilisez les fonctions pandas nécessaires. Ne pas chercher les réponses à ces questions à la main, ni en utilisant une IA de quelque façon que ce soit...

Parmi les événements mentionnés dans cette page, combien y a-t-il de concerts ? Combien de "comédies" ?

(Si vous utilisez le répertoire cepac_silo_pages, vous devriez trouver respectivement 40 et 29).


#### Q16

Combien d\'événements / dates pour lesquels le prix n\'est pas mentionné ? 

(Exceptionnellement, vous pouvez répondre à la question juste avec un affichage, pas besoin d'écrire une fonction pour retrouver cela).

In [21]:
concert = evenements[
    evenements["categorie"].str.contains(
        "Concert",
        case=False,
        na=False
    )
]


comedie = evenements[
    evenements["categorie"].str.contains(
        "Humour|One man show|One-man-show",
        case=False,
        na=False
    )
]





In [22]:
display(concert)


,titre,categorie,categorie_2,prix_categorie_2,categorie_1,prix_categorie_1,categorie_or,prix_categorie_or,jour_1,heure_1,jour_2,heure_2,jour_3,heure_3
0,DEVIN TOWNSEND,Concert,CATEGORIE 2,63.0,CATEGORIE 1,74.0,CATEGORIE OR,90.5,mercredi,20h00,None,None,None,None
1,The bootleg beatles,Concert,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None,None
3,LETZ ZEP,Concert,CATEGORIE 2,63.0,CATEGORIE 1,74.0,CATEGORIE OR,90.5,mercredi,20h00,None,None,None,None
4,THE SISTERS OF MERCY,Concert,NaN,NaN,NaN,NaN,NaN,NaN,jeudi,20h00,None,None,None,None
7,TIF,Concert - Rap / Urbain,NaN,NaN,NaN,NaN,NaN,NaN,jeudi,20h00,None,None,None,None
8,Boudchart,Concert,CATEGORIE 2,45.0,CATEGORIE 1,55.0,CATEGORIE OR,65.0,vendredi,20h00,None,None,None,None
9,WORLDS APART,Concert,CATEGORIE 2,49.0,CATEGORIE 1,59.0,CATEGORIE OR,NaN,samedi,20h00,None,None,None,None
11,TINA ARENA,Concert,CATEGORIE 2,49.0,CATEGORIE 1,59.0,CATEGORIE OR,69.0,mercredi,20h00,None,None,None,None
12,THE VOICE OF GEORGE MICHAEL,Concert,NaN,NaN,NaN,NaN,NaN,NaN,jeudi,20h30,jeudi,20h30,None,None
15,Bachar Mar-Khalifé joue Christophe,Concert,NaN,NaN,NaN,NaN,NaN,NaN,vendredi,21h00,None,None,None,None


In [26]:
print(len(concert))

43


In [ ]:
display(comedie)


In [27]:
print(len(comedie))

33


In [24]:
sans_prix = evenements[
    evenements["prix_categorie_2"].isna()
    & evenements["prix_categorie_1"].isna()
    & evenements["prix_categorie_or"].isna()
]

display(len(sans_prix))

22

#### Q17
Parmi les événements mentionnés sur la page, quel est le jour de la semaine le plus fréquent (ne pas le chercher à la main, utiliser des fonctions pandas pour retrouver cette valeur)?

In [25]:
jours = pd.concat([
    evenements["jour_1"],
    evenements["jour_2"],
    evenements["jour_3"]
])

frequence_jours = jours.value_counts()

jour_plus_frequent = frequence_jours.idxmax()

print(
    "Le jour de la semaine le plus fréquent est :",
    jour_plus_frequent
)

Le jour de la semaine le plus fréquent est : samedi


#### Q18
On recherche maintenant les prix moyens des événements. 

On ignorera les prix à None. C'est-à-dire qu'on les mettra à nan de manière à ce qu'elles ne soient pas prises en compte dans le calcul de la moyenne...

Quel est le prix moyen d'une place pour un concert (ne pas le chercher à la main, utiliser des fonctions pandas voire numpy pour retrouver cette valeur) ? 

Quel est le prix moyen d'une place pour une comédie (ne pas le chercher à la main, utiliser des fonctions pandas voire numpy pour retrouver cette valeur) ? 

In [28]:
import numpy as np


concert = concert.replace({None: np.nan})
comedie = comedie.replace({None: np.nan})


colonnes_prix = [
    "prix_categorie_2",
    "prix_categorie_1",
    "prix_categorie_or"
]


prix_moyen_concert = concert[colonnes_prix].stack().mean()


prix_moyen_comedie = comedie[colonnes_prix].stack().mean()


print(
    "Prix moyen d'une place pour un concert :",
    prix_moyen_concert,
    "€"
)

print(
    "Prix moyen d'une place pour une comédie :",
    prix_moyen_comedie,
    "€"
)

Prix moyen d'une place pour un concert : 54.526436781609185 €
Prix moyen d'une place pour une comédie : 40.74833333333333 €


#### Q19

## <font color='red'> Point de validation Bonus : une fois que vous aurez terminé cette question, appelez la prof et attendez-vous à répondre à des questions sur votre code <img src="buttonValidate.gif"> 
    
Menez à bien une analyse exploratoire de ces données, et une visualisation de ces données, qui soit la plus intéressante possible. Vous êtes libres du choix des librairies, mais vous devrez être capables d'expliquer, ligne par ligne, votre code...
